# CIFAR-10 Sparse Labeling with Vanilla CNN

This notebook demonstrates sparse labeling on CIFAR-10 using a simple CNN. This is a common scenario in federated learning where different clients have different amounts of labeled data.

## Overview
- Load CIFAR-10 dataset with artificial sparse labeling (only 10% labeled)
- Visualize labeled vs unlabeled samples
- Train a vanilla CNN using only labeled samples
- Evaluate performance and analyze results

## 1. Setup and Imports

First, let's import the necessary libraries and our custom classes.

In [ ]:
# Standard imports
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Optional

# Add the examples directory to path
sys.path.append('../examples')

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Import our custom classes
from sparse_cifar10_example import SparseCIFAR10Dataset, VanillaCNN

print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 2. Configuration and Dataset Creation

Let's set up our configuration and create sparse datasets with different sparsity levels.

In [ ]:
# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 32
sparsity = 0.1  # Only 10% of training data will be labeled
num_epochs = 5

# Data transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print(f"Configuration:")
print(f"  Device: {device}")
print(f"  Batch size: {batch_size}")
print(f"  Sparsity: {sparsity*100}% labeled")
print(f"  Epochs: {num_epochs}")

In [ ]:
# Create datasets
print("Creating sparse CIFAR-10 datasets...")

train_dataset = SparseCIFAR10Dataset(
    root='./data', 
    train=True, 
    sparsity=sparsity, 
    transform=transform, 
    download=True
)

test_dataset = SparseCIFAR10Dataset(
    root='./data', 
    train=False, 
    sparsity=1.0,  # Test set fully labeled for evaluation
    transform=transform, 
    download=False
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nDataset sizes:")
print(f"  Training: {len(train_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")
print(f"  Labeled training samples: ~{int(len(train_dataset) * sparsity)}")

## 3. Data Visualization

Let's visualize some examples of labeled and unlabeled samples to understand our sparse dataset.

In [ ]:
# CIFAR-10 class names
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

def visualize_sparse_samples(dataset, num_samples=12, cols=4):
    """Visualize labeled and unlabeled samples."""
    
    rows = (num_samples + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    sample_count = 0
    
    # Find examples of both labeled and unlabeled samples
    for idx in range(len(dataset)):
        if sample_count >= num_samples:
            break
            
        image, label, is_labeled = dataset[idx]
        
        # Convert tensor to numpy for plotting
        image_np = image.permute(1, 2, 0).numpy()
        image_np = (image_np + 1) / 2  # Denormalize
        image_np = np.clip(image_np, 0, 1)
        
        row = sample_count // cols
        col = sample_count % cols
        
        axes[row, col].imshow(image_np)
        
        # Create title
        status = "Labeled" if is_labeled else "Unlabeled"
        class_name = classes[label] if label != -1 else "Unknown"
        title = f"{status}: {class_name}"
        
        # Color code the title
        color = 'green' if is_labeled else 'red'
        axes[row, col].set_title(title, fontsize=12, color=color, fontweight='bold')
        axes[row, col].axis('off')
        
        sample_count += 1
    
    # Hide any unused subplots
    for idx in range(sample_count, rows * cols):
        row = idx // cols
        col = idx % cols
        axes[row, col].axis('off')
    
    plt.suptitle(f'CIFAR-10 Sparse Dataset Samples (Sparsity: {sparsity*100}%)', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize samples
visualize_sparse_samples(train_dataset, num_samples=12)

## 4. Analyze Label Distribution

Let's analyze how the sparse labeling affects the class distribution.

In [ ]:
# Analyze label distribution
labeled_counts = {i: 0 for i in range(10)}
unlabeled_count = 0
total_labeled = 0

for idx in range(len(train_dataset)):
    _, label, is_labeled = train_dataset[idx]
    
    if is_labeled:
        labeled_counts[label] += 1
        total_labeled += 1
    else:
        unlabeled_count += 1

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot of labeled samples per class
class_names = [classes[i] for i in range(10)]
counts = [labeled_counts[i] for i in range(10)]

bars = ax1.bar(class_names, counts, color='skyblue', edgecolor='navy', alpha=0.7)
ax1.set_title('Labeled Samples per Class', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Labeled Samples')
ax1.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 10,
             f'{int(height)}', ha='center', va='bottom')

# Pie chart of labeled vs unlabeled
labels = ['Labeled', 'Unlabeled']
sizes = [total_labeled, unlabeled_count]
colors = ['lightgreen', 'lightcoral']
explode = (0.05, 0)  # slightly separate the labeled slice

wedges, texts, autotexts = ax2.pie(sizes, explode=explode, labels=labels, colors=colors,
                                   autopct='%1.1f%%', shadow=True, startangle=90)
ax2.set_title(f'Label Distribution\n(Target: {sparsity*100}% labeled)', 
              fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nLabel Distribution Summary:")
print(f"  Total samples: {len(train_dataset)}")
print(f"  Labeled: {total_labeled} ({total_labeled/len(train_dataset)*100:.1f}%)")
print(f"  Unlabeled: {unlabeled_count} ({unlabeled_count/len(train_dataset)*100:.1f}%)")
print(f"  Average labeled per class: {total_labeled/10:.1f}")

## 5. Model Creation and Architecture

Let's create our vanilla CNN and examine its architecture.

In [ ]:
# Create model
model = VanillaCNN(num_classes=10).to(device)

# Print model architecture
print("Model Architecture:")
print("=" * 50)
print(model)
print("=" * 50)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size (MB): {total_params * 4 / (1024**2):.2f}")

## 6. Training Loop

Now let's train our model using only the labeled samples.

In [ ]:
# Training setup
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Track training progress
train_losses = []
train_accuracies = []
labeled_samples_per_epoch = []

print("Starting training...")
print("=" * 60)

model.train()

for epoch in range(num_epochs):
    epoch_loss = 0.0
    epoch_correct = 0
    epoch_labeled = 0
    
    for batch_idx, (data, targets, is_labeled) in enumerate(train_loader):
        data = data.to(device)
        targets = targets.to(device)
        is_labeled = is_labeled.to(device)
        
        # Filter only labeled samples
        labeled_mask = is_labeled & (targets != -1)
        
        if labeled_mask.sum() == 0:
            continue  # Skip batch if no labeled samples
            
        labeled_data = data[labeled_mask]
        labeled_targets = targets[labeled_mask]
        
        optimizer.zero_grad()
        outputs = model(labeled_data)
        loss = criterion(outputs, labeled_targets)
        loss.backward()
        optimizer.step()
        
        # Track statistics
        epoch_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        epoch_correct += (predicted == labeled_targets).sum().item()
        epoch_labeled += labeled_mask.sum().item()
        
        # Print progress every 100 batches
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx:3d}: Loss={loss.item():.4f}, '
                  f'Labeled samples={labeled_mask.sum().item():2d}')
    
    # Calculate epoch metrics
    avg_loss = epoch_loss / len(train_loader)
    accuracy = 100.0 * epoch_correct / max(1, epoch_labeled) if epoch_labeled > 0 else 0
    
    train_losses.append(avg_loss)
    train_accuracies.append(accuracy)
    labeled_samples_per_epoch.append(epoch_labeled)
    
    print(f"\nEpoch {epoch+1}/{num_epochs}:")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Training Accuracy: {accuracy:.2f}%")
    print(f"  Labeled samples used: {epoch_labeled}")
    print("-" * 40)

print("\nTraining completed!")

## 7. Training Visualization

Let's visualize the training progress.

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss curve
epochs = range(1, num_epochs + 1)
ax1.plot(epochs, train_losses, 'b-o', linewidth=2, markersize=6)
ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Average Loss')
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(epochs, train_accuracies, 'g-o', linewidth=2, markersize=6)
ax2.set_title('Training Accuracy (on labeled data)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final training stats
print(f"\nFinal Training Statistics:")
print(f"  Final Loss: {train_losses[-1]:.4f}")
print(f"  Final Training Accuracy: {train_accuracies[-1]:.2f}%")
print(f"  Total labeled samples used per epoch: {labeled_samples_per_epoch[-1]}")
print(f"  Utilization of available labeled data: {labeled_samples_per_epoch[-1]/total_labeled*100:.1f}%")

## 8. Model Evaluation

Now let's evaluate our trained model on the full test set.

In [ ]:
# Evaluate on test set
model.eval()
correct = 0
total = 0
class_correct = {i: 0 for i in range(10)}
class_total = {i: 0 for i in range(10)}

with torch.no_grad():
    for data, targets, _ in test_loader:  # Ignore is_labeled for test
        data = data.to(device)
        targets = targets.to(device)
        
        outputs = model(data)
        _, predicted = torch.max(outputs, 1)
        
        total += targets.size(0)
        correct += (predicted == targets).sum().item()
        
        # Per-class accuracy
        for i in range(targets.size(0)):
            label = targets[i].item()
            class_total[label] += 1
            if predicted[i] == targets[i]:
                class_correct[label] += 1

# Calculate accuracies
overall_accuracy = 100 * correct / total
class_accuracies = {i: 100 * class_correct[i] / max(1, class_total[i]) for i in range(10)}

print(f"\nTest Set Evaluation Results:")
print("=" * 50)
print(f"Overall Accuracy: {overall_accuracy:.2f}% ({correct}/{total})")
print("\nPer-class Accuracy:")
print("-" * 30)

for i in range(10):
    print(f"  {classes[i]:8s}: {class_accuracies[i]:5.1f}% ({class_correct[i]:3d}/{class_total[i]:3d})")

print(f"\nAverage per-class accuracy: {np.mean(list(class_accuracies.values())):.2f}%")

## 9. Results Visualization

Let's create visualizations to better understand our model's performance.

In [ ]:
# Create comprehensive results visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. Per-class accuracy bar plot
class_names = [classes[i] for i in range(10)]
accuracies = [class_accuracies[i] for i in range(10)]
colors = plt.cm.Set3(np.linspace(0, 1, 10))

bars = ax1.bar(class_names, accuracies, color=colors, edgecolor='black', alpha=0.8)
ax1.set_title('Per-Class Test Accuracy', fontsize=14, fontweight='bold')
ax1.set_ylabel('Accuracy (%)')
ax1.tick_params(axis='x', rotation=45)
ax1.axhline(y=overall_accuracy, color='red', linestyle='--', 
            label=f'Overall: {overall_accuracy:.1f}%')
ax1.legend()

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{height:.1f}%', ha='center', va='bottom', fontsize=10)

# 2. Training vs Test comparison
metrics = ['Training Accuracy\n(Labeled Data)', 'Test Accuracy\n(All Data)']
values = [train_accuracies[-1], overall_accuracy]
colors_comp = ['lightblue', 'lightcoral']

bars_comp = ax2.bar(metrics, values, color=colors_comp, edgecolor='black', alpha=0.8)
ax2.set_title('Training vs Test Performance', fontsize=14, fontweight='bold')
ax2.set_ylabel('Accuracy (%)')

for i, bar in enumerate(bars_comp):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{height:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

# 3. Data utilization pie chart
data_labels = [f'Labeled\n({total_labeled})', f'Unlabeled\n({unlabeled_count})']
data_sizes = [total_labeled, unlabeled_count]
data_colors = ['lightgreen', 'lightgray']

wedges, texts, autotexts = ax3.pie(data_sizes, labels=data_labels, colors=data_colors,
                                   autopct='%1.1f%%', shadow=True, startangle=90)
ax3.set_title(f'Data Utilization\n(Sparsity: {sparsity*100}%)', 
              fontsize=14, fontweight='bold')

# 4. Model performance summary
ax4.axis('off')
summary_text = f"""
📊 EXPERIMENT SUMMARY

🔧 Configuration:
  • Dataset: CIFAR-10
  • Sparsity: {sparsity*100}% labeled ({total_labeled:,} samples)
  • Model: Vanilla CNN ({total_params:,} parameters)
  • Training epochs: {num_epochs}
  • Device: {device}

📈 Results:
  • Test Accuracy: {overall_accuracy:.2f}%
  • Training Accuracy: {train_accuracies[-1]:.2f}%
  • Final Loss: {train_losses[-1]:.4f}
  • Best Class: {classes[np.argmax(accuracies)]} ({max(accuracies):.1f}%)
  • Worst Class: {classes[np.argmin(accuracies)]} ({min(accuracies):.1f}%)

💡 Insights:
  • Model trained with only {sparsity*100}% of labels
  • Achieved {overall_accuracy:.1f}% accuracy on full test set
  • Performance gap: {abs(train_accuracies[-1] - overall_accuracy):.1f}% points
  • Ready for federated learning scenarios!
"""

ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes, fontsize=11,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

## 10. Sample Predictions

Let's look at some individual predictions to understand how well our model performs.

In [ ]:
# Show sample predictions
def show_predictions(model, dataset, num_samples=8):
    """Show model predictions on sample images."""
    
    model.eval()
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    # Get random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, true_label, _ = dataset[idx]
            
            # Get prediction
            image_batch = image.unsqueeze(0).to(device)
            output = model(image_batch)
            _, predicted = torch.max(output, 1)
            predicted_label = predicted.item()
            
            # Get prediction confidence
            probabilities = F.softmax(output, dim=1)
            confidence = probabilities[0][predicted_label].item() * 100
            
            # Display image
            image_np = image.permute(1, 2, 0).numpy()
            image_np = (image_np + 1) / 2  # Denormalize
            image_np = np.clip(image_np, 0, 1)
            
            axes[i].imshow(image_np)
            
            # Create title with prediction info
            true_class = classes[true_label]
            pred_class = classes[predicted_label]
            
            if true_label == predicted_label:
                title = f"✅ {pred_class}\n({confidence:.1f}%)"
                color = 'green'
            else:
                title = f"❌ True: {true_class}\nPred: {pred_class} ({confidence:.1f}%)"
                color = 'red'
            
            axes[i].set_title(title, fontsize=10, color=color, fontweight='bold')
            axes[i].axis('off')
    
    plt.suptitle('Sample Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(model, test_dataset, num_samples=8)

## 11. Conclusions and Next Steps

Let's summarize our findings and discuss potential improvements.

In [ ]:
# Final analysis
print("🎯 EXPERIMENT CONCLUSIONS")
print("=" * 60)

print(f"\n📊 Performance Summary:")
print(f"  • Successfully trained CNN with only {sparsity*100}% labeled data")
print(f"  • Achieved {overall_accuracy:.2f}% accuracy on test set")
print(f"  • Used {total_labeled:,} labeled samples out of {len(train_dataset):,} total")
print(f"  • Model has {total_params:,} parameters")

print(f"\n🔍 Key Observations:")
print(f"  • Training accuracy: {train_accuracies[-1]:.1f}%")
print(f"  • Test accuracy: {overall_accuracy:.1f}%")
print(f"  • Generalization gap: {abs(train_accuracies[-1] - overall_accuracy):.1f} percentage points")
print(f"  • Best performing class: {classes[np.argmax(accuracies)]} ({max(accuracies):.1f}%)")
print(f"  • Most challenging class: {classes[np.argmin(accuracies)]} ({min(accuracies):.1f}%)")

print(f"\n🚀 Federated Learning Applications:")
print(f"  • This approach simulates federated learning scenarios")
print(f"  • Different clients could have different sparsity levels")
print(f"  • Semi-supervised techniques could improve performance")
print(f"  • Ready for multi-client federated averaging")

print(f"\n💡 Potential Improvements:")
print(f"  • Data augmentation for better generalization")
print(f"  • Semi-supervised learning with unlabeled data")
print(f"  • Regularization techniques (dropout, weight decay)")
print(f"  • Transfer learning from pre-trained models")
print(f"  • Federated optimization algorithms (FedAvg, FedProx)")

# Compare to baseline (random guessing)
random_accuracy = 100 / 10  # 10% for 10 classes
improvement = overall_accuracy - random_accuracy

print(f"\n📈 Performance Context:")
print(f"  • Random guessing baseline: {random_accuracy:.1f}%")
print(f"  • Our model improvement: +{improvement:.1f} percentage points")
print(f"  • Relative improvement: {improvement/random_accuracy*100:.1f}% over random")

print("\n" + "=" * 60)
print("✨ Experiment completed successfully! Ready for federated learning! ✨")
print("=" * 60)